# E-commerce Data Engineering Pipeline | 100K Orders

**Bronze → Silver → Gold** using Python, Pandas, SQLite, SQL, quality controls, and reporting. The bundled dataset is synthetic.

## 1. Load project into Google Colab
Upload `ecommerce-etl-pipeline.zip` from your computer when prompted. If you have cloned the GitHub repo, skip the upload and set `PROJECT` to the cloned folder.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # select ecommerce-etl-pipeline.zip
zip_name = next((n for n in uploaded if n.endswith('.zip')), None)
assert zip_name, 'Please upload the project ZIP file.'
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')
PROJECT = Path('/content/ecommerce-etl-pipeline')
assert (PROJECT / 'src/pipeline.py').exists()
print('Project ready:', PROJECT)

## 2. Install dependencies and inspect raw data

In [ ]:
%pip -q install -r /content/ecommerce-etl-pipeline/requirements.txt
import sys
sys.path.insert(0, str(PROJECT))
import pandas as pd
from src.pipeline import extract, transform, load, run
orders, customers, products = extract(PROJECT / 'data/raw')
print('Orders:',len(orders),'Customers:',len(customers),'Products:',len(products))
display(orders.head())

## 3. Run the ETL and inspect data quality

In [ ]:
quality, monthly, categories, states = run(PROJECT/'data/raw', PROJECT/'outputs')
display(pd.DataFrame([quality]).T.rename(columns={0:'value'}))
assert quality['all_rows_accounted_for']
assert quality['accepted_rows'] == 99375

## 4. Explore the Silver layer and rejected records

In [ ]:
silver = pd.read_csv(PROJECT/'outputs/silver_orders.csv')
rejected = pd.read_csv(PROJECT/'outputs/rejected_orders.csv')
print('Accepted:', len(silver), 'Quarantined:', len(rejected))
display(silver.head()); display(rejected.head())

## 5. Query your SQLite data warehouse using SQL

In [ ]:
import sqlite3
with sqlite3.connect(PROJECT/'outputs/ecommerce.db') as con:
    sql_report = pd.read_sql_query('''SELECT category, COUNT(*) AS orders, ROUND(SUM(recognized_revenue), 2) AS revenue FROM silver_orders GROUP BY category ORDER BY revenue DESC''', con)
display(sql_report)

## 6. Plot revenue by month and by category

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(monthly['order_month'], monthly['revenue'], marker='o')
ax.set(title='Recognized revenue by month', xlabel='Month', ylabel='Revenue ($)')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(categories['category'], categories['revenue'])
ax.set(title='Revenue by category', ylabel='Revenue ($)')
plt.tight_layout(); plt.show()

## 7. Run automated tests

In [ ]:
import subprocess
subprocess.run(['python','-m','pytest','-q',str(PROJECT/'tests')], cwd=str(PROJECT), check=True)

## 8. Download output (optional)
Outputs in Colab runtime disappear when your session resets. You may download the output package or copy it to your Google Drive.

In [ ]:
from google.colab import files
import shutil
archive = shutil.make_archive('/content/ecommerce-etl-results', 'zip', PROJECT/'outputs')
files.download(archive)